# GalaxyPPG PPG inversion check

This notebook visualizes the raw Galaxy Watch PPG waveform and the canonical inverted waveform used by the corrected pipeline. The loader performs the inversion exactly once and keeps both `ppg_raw` and `ppg` for auditability.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from src.data import load_participant_data

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'GalaxyPPG'
participant_id = 'P02'
participant = load_participant_data(participant_id, dataset_root=DATASET_ROOT, reference_source='ibi')

participant.ppg.head()

In [ ]:
participant.ppg[['ppg_raw', 'ppg', 'ppg_inverted', 'ppg_canonical_source']].head(3)

In [ ]:
events = participant.events
sessions = []
starts = {}
for row in events.sort_values('timestamp_ms').itertuples(index=False):
    if row.status == 'ENTER':
        starts[row.session] = int(row.timestamp_ms)
    elif row.status == 'EXIT' and row.session in starts:
        start_ms = starts.pop(row.session)
        end_ms = int(row.timestamp_ms)
        if end_ms > start_ms:
            sessions.append((row.session, start_ms, end_ms))

sessions[:5]

In [ ]:
def plot_session_ppg(session_name: str, start_ms: int, end_ms: int, seconds: int = 20) -> None:
    segment_end_ms = min(end_ms, start_ms + seconds * 1000)
    segment = participant.ppg[
        (participant.ppg['timestamp_ms'] >= start_ms)
        & (participant.ppg['timestamp_ms'] < segment_end_ms)
    ].copy()
    if segment.empty:
        print(f'No PPG samples for {session_name}')
        return
    segment['time_s'] = (segment['timestamp_ms'] - segment['timestamp_ms'].iloc[0]) / 1000.0
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(segment['time_s'], segment['ppg_raw'], color='tab:gray', linewidth=1)
    axes[0].set_title(f'{participant_id} {session_name}: raw Galaxy Watch PPG')
    axes[0].set_ylabel('raw PPG')
    axes[1].plot(segment['time_s'], segment['ppg'], color='tab:blue', linewidth=1)
    axes[1].set_title('canonical inverted PPG used downstream')
    axes[1].set_xlabel('seconds')
    axes[1].set_ylabel('canonical PPG')
    fig.tight_layout()
    plt.show()

for session_name, start_ms, end_ms in sessions[:3]:
    plot_session_ppg(session_name, start_ms, end_ms, seconds=20)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sample = participant.ppg.iloc[:2000]
ax.scatter(sample['ppg_raw'], sample['ppg'], s=4, alpha=0.4)
ax.set_xlabel('ppg_raw')
ax.set_ylabel('canonical ppg')
ax.set_title('Loader invariant: canonical ppg = -ppg_raw')
ax.grid(True, alpha=0.3)
plt.show()